<a href="https://colab.research.google.com/github/karolinajapinska/E-commerce_accountdynamic-analysis/blob/main/Practical_Challenge_2_Supply_Chain_BI_Delivery_Delay_Optimization_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Business Objective

The company has observed a significant decrease in On-Time Delivery (OTD) performance, resulting in customer dissatisfaction and an increasing number of complaints.

The goal of this project is to build an analytical dashboard that helps logistics managers:

* Measure the scale of delivery delays.
* Identify where delays occur most frequently.
* Understand which factors contribute to delays.
* Estimate the financial impact of delayed deliveries.
* Prioritize corrective actions based on business impact.

This project focuses on business analytics and root cause analysis, not predictive modeling.

#2. ETL Process
##2.1 Import libraries

In [ ]:
import kagglehub
import pandas as pd
import os
from google.colab import drive

##2.2 Extract

Download and load the DataCo Smart Supply Chain dataset.

In [ ]:
path = kagglehub.dataset_download(
    "shashwatwork/dataco-smart-supply-chain-for-big-data-analysis"
)

df = pd.read_csv(
    os.path.join(path, "DataCoSupplyChainDataset.csv"),
    encoding="latin1"
)
df.head()

Using Colab cache for faster access to the 'dataco-smart-supply-chain-for-big-data-analysis' dataset.


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


##2.3 Data Quality Validation

Before analysis, validate dataset quality.

In [ ]:
#Dataset Structure
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

(180519, 53)

In [ ]:
# Missing Values
df.isna().sum().sort_values(ascending=False)

,0
Product Description,180519
Order Zipcode,155679
Customer Lname,8
Customer Zipcode,3
Days for shipment (scheduled),0
Sales per customer,0
Benefit per order,0
Delivery Status,0
Late_delivery_risk,0
Customer City,0


In [ ]:
# Duplicate Records
df.duplicated().sum()
df = df.drop_duplicates()
df.shape

(180519, 53)

In [ ]:
#Data Types Validation
#Check:
#Date columns, Numerical columns, Categorical columns
df.dtypes

,0
Type,object
Days for shipping (real),int64
Days for shipment (scheduled),int64
Benefit per order,float64
Sales per customer,float64
Delivery Status,object
Late_delivery_risk,int64
Category Id,int64
Category Name,object
Customer City,object


#2.4 Remove Irrelevant Columns
## Column Relevance Assessment

The objective of this project is to identify the root causes of delayed deliveries and quantify their operational and financial impact.

To improve analytical efficiency and dashboard usability, each column was evaluated according to its relevance for:

* Delivery performance analysis
* Geographic analysis
* Product analysis
* Customer segmentation
* Financial impact assessment

Columns containing sensitive information, duplicate information, technical identifiers without analytical value, or fields unrelated to logistics performance were excluded from the final analytical dataset.


In [ ]:
column_review = pd.DataFrame({
    "Column": df.columns
})

column_review.head(53)

,Column
0,Type
1,Days for shipping (real)
2,Days for shipment (scheduled)
3,Benefit per order
4,Sales per customer
5,Delivery Status
6,Late_delivery_risk
7,Category Id
8,Category Name
9,Customer City


##Group A — Core Delivery Analysis (KEEP)

These are mandatory.
| Column                        | Decision | Reason                      |
| ----------------------------- | -------- | --------------------------- |
| Delivery Status               | Keep     | Main delivery outcome       |
| Late_delivery_risk            | Keep     | Existing business indicator |
| Days for shipping (real)      | Keep     | Actual delivery duration    |
| Days for shipment (scheduled) | Keep     | Planned delivery duration   |
| Shipping Mode                 | Keep     | Root cause dimension        |
| Order Status                  | Keep     | Order lifecycle analysis    |

### Delivery Performance Variables

These variables directly measure delivery execution and are the primary source for calculating delay-related KPIs.

The difference between actual and scheduled shipping duration forms the foundation of the delay analysis performed throughout the project.

##Group B — Geographic Analysis (KEEP)

Questions:
Where do delays happen?
Which regions underperform?
| Column        | Decision | Reason                 |
| ------------- | -------- | ---------------------- |
| Market        | Keep     | Global market analysis |
| Order Region  | Keep     | Regional comparison    |
| Order Country | Keep     | Country comparison     |
| Order City    | Keep     | Detailed investigation |
### Geographic Variables

Geographic dimensions are essential for identifying operational bottlenecks across markets, regions, countries, and cities.

These variables enable drill-down analysis from global performance to specific locations where delays are concentrated.
##Group C — Product Analysis (KEEP)

Questions:
Which products are associated with delays?
| Column          | Decision | Reason                      |
| --------------- | -------- | --------------------------- |
| Category Name   | Keep     | Product category analysis   |
| Product Name    | Keep     | Product-level investigation |
| Department Name | Keep     | Product grouping            |
##Group D — Financial Impact (KEEP)

Questions:
Where does the company lose money?
| Column                 | Decision | Reason           |
| ---------------------- | -------- | ---------------- |
| Sales                  | Keep     | Revenue analysis |
| Order Item Total       | Keep     | Order value      |
| Benefit per order      | Keep     | Profitability    |
| Order Profit Per Order | Keep     | Financial impact |
### Financial Variables

Operational delays become strategically important when they affect revenue and profitability.

Financial measures were retained to quantify the business impact of delayed deliveries and prioritize corrective actions based on monetary exposure.
##Group E — Customer Segmentation (KEEP)

Questions:
Which customers are most affected?
| Column           | Decision | Reason                 |
| ---------------- | -------- | ---------------------- |
| Customer Segment | Keep     | Segment comparison     |
| Customer Id      | Keep     | Customer-level metrics |
##Group F — Date Variables (KEEP)
| Column        | Decision | Reason           |
| ------------- | -------- | ---------------- |
| order_date    | Keep     | Trend analysis   |
| shipping_date | Keep     | Process analysis |
##Group G — Technical Keys (KEEP)
| Column          | Decision |
| --------------- | -------- |
| Order Id        | Keep     |
| Order Item Id   | Keep     |
| Product Card Id | Keep     |
| Customer Id     | Keep     |
##Group H — Personal Information (REMOVE)
| Column            | Decision | Reason              |
| ----------------- | -------- | ------------------- |
| Customer Email    | Remove   | Sensitive data      |
| Customer Password | Remove   | Sensitive data      |
| Customer Fname    | Remove   | No analytical value |
| Customer Lname    | Remove   | No analytical value |
| Customer Street   | Remove   | Too granular        |
### Personally Identifiable Information (PII)

Columns containing personally identifiable information were removed because they do not contribute to delivery performance analysis and may introduce unnecessary privacy risks.

Examples include customer names, email addresses, passwords, and street-level addresses.
##Group I — Geographic Redundancy (REMOVE)
| Column           | Decision | Reason |
| ---------------- | -------- | ------ |
| Latitude         | Remove   |        |
| Longitude        | Remove   |        |
| Customer Zipcode | Remove   |        |
Reason: Region, Country, City already provide sufficient geographic detail.
##Group J — Descriptive Text Fields (REMOVE)
| Column              | Decision | Reason |
| ------------------- | -------- | ------ |
| Product Description | Remove   |        |
#Final Dataset Recommendation

For a Tableau dashboard I would end up with approximately:
###Dimensions:
Order Date
Shipping Date
Market
Region
Country
City
Shipping Mode
Customer Segment
Category Name
Department Name
Product Name
###Measures:
Sales
Profit
Order Item Total
Delay Days
Late Sales Value
Late Profit Impact
###Keys:
Order Id
Order Item Id
Customer Id
Product Card Id
###Calculated Fields:
Delay Days
Is Delayed
Delivery Performance
Delay Severity
Late Sales Value
Late Profit Impact
Delay Rate
On-Time Delivery Rate

##2.5 Data Framework clean-up

In [ ]:
columns_to_remove = [
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Product Description',
    'Latitude',
    'Longitude'
]

df = df.drop(
    columns=[col for col in columns_to_remove if col in df.columns]
)

print(df.shape)

(180519, 44)


##2.6 Data Transformation

Convert dates:

In [ ]:
df = df.rename(columns={
    'order date (DateOrders)': 'order_date'
    })
df = df.rename(columns={
    'shipping date (DateOrders)': 'shipping_date'
    })


print(f"Columns after advanced cleaning: {df.columns.tolist()}") # Debugging line

df['order_date'] = pd.to_datetime(
    df['order_date'],
    errors='coerce'
)

df['shipping_date'] = pd.to_datetime(
    df['shipping_date'],
    errors='coerce'
)
# Validate chronological order:
invalid_dates = df[df['shipping_date'] < df['order_date']]
if invalid_dates.empty:
    print("Validation passed: no records found where shipping_date is earlier than order_date.")
else:
    print(f"Validation failed: {len(invalid_dates)} records have shipping_date earlier than order_date.")

Columns after advanced cleaning: ['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Department Id', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order_date', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping_date', 'Shipping Mode']
Validation passed: no records found where shipping_date is earlier than order_date.


#2.7 Create Analytical Variables
Core business metric.

In [ ]:
df['delay_days'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Delay flag
df['is_delayed'] = (
    df['delay_days'] > 0
).astype(int)

# Delivery Performance Category
def delivery_status(x):
    if x > 0:
        return "Delayed"
    elif x == 0:
        return "On Time"
    else:
        return "Early"

df['delivery_performance'] = (
    df['delay_days']
    .apply(delivery_status)
)

#Delay Severity
def delay_severity(x):

    if x <= 0:
        return "No Delay"

    elif x <= 1:
        return "Minor Delay"

    elif x <= 3:
        return "Moderate Delay"

    else:
        return "Severe Delay"

df['delay_severity'] = (
    df['delay_days']
    .apply(delay_severity)
)

# Delayed Sales Value

df['late_sales_value'] = (
    df['Sales']
    * df['is_delayed']
)

# Delayed Profit Impact
df['late_profit_impact'] = (
    df['Order Profit Per Order']
    * df['is_delayed']
)

df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Product Price,Product Status,shipping_date,Shipping Mode,delay_days,is_delayed,delivery_performance,delay_severity,late_sales_value,late_profit_impact
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,327.75,0,2018-02-03 22:56:00,Standard Class,-1,0,Early,No Delay,0.00,0.000000
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,327.75,0,2018-01-18 12:27:00,Standard Class,1,1,Delayed,Minor Delay,327.75,-249.089996
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,327.75,0,2018-01-17 12:06:00,Standard Class,0,0,On Time,No Delay,0.00,-0.000000
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,327.75,0,2018-01-16 11:45:00,Standard Class,-1,0,Early,No Delay,0.00,0.000000
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,327.75,0,2018-01-15 11:24:00,Standard Class,-2,0,Early,No Delay,0.00,0.000000


#3. Export final Excel file
The final Excel file will be used as a data source in Tableau.

In [ ]:
from google.colab import drive
!pip install openpyxl
import pandas as pd # Re-import pandas to ensure it picks up openpyxl

if not df.empty:

    # Mount Google Drive
    drive.mount('/content/drive')

    # Save a copy to Google Drive
    drive_output_path = '/content/drive/MyDrive/Mate_homework/delivery_delay_optimization.xlsx'
    df.to_excel(drive_output_path, index=False)

    print(f'Excel saved to: {drive_output_path}')

else:
    print("final_results DataFrame is empty. Skipping export.")

Mounted at /content/drive
Excel saved to: /content/drive/MyDrive/Mate_homework/delivery_delay_optimization.xlsx


#4. Tableau Visualization
link: https://public.tableau.com/views/SupplyChainBIDeliveryDelayOptimizationChallenge/RootCauseAnalysis?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

#5. Business Conclusions
##5.1 Business Observations
* Delivery performance is a significant challenge across the supply chain. More than half of all orders (57.33%) were delivered late, resulting in an On-Time Delivery Rate of only 42.67%. This indicates a systemic issue rather than isolated incidents.
* Delay rates remained consistently high throughout the analyzed period, fluctuating between approximately 56% and 59%. The absence of a sustained downward trend suggests that the company has not effectively addressed the root causes of delivery delays over time.
* Central Africa (60.61%), East Africa (58.51%), and South of USA (58.21%) recorded the highest delay rates among all regions. These regions should be considered operational hotspots requiring immediate attention.
* Shipping mode has a substantial impact on delivery performance. First Class shipments show a 100% delay rate, while Second Class shipments reach nearly 80%. In contrast, Standard Class performs significantly better with a delay rate below 40%.
* Product categories exhibit considerable variation in delay performance. Categories such as Golf Bags & Carts, Lacrosse, and Cameras experience the highest delay rates, suggesting potential inventory, warehousing, or transportation bottlenecks associated with these products.
* The financial impact analysis reveals that Western Europe and Central America generate the highest value of delayed sales, exceeding 3 million PLN each. Although some regions do not have the highest delay percentages, they represent the greatest financial exposure due to their larger sales volumes.
##5.2 Business Recommendations
* Prioritize improvement initiatives in Western Europe and Central America, as these regions contribute the largest share of delayed sales value. Reducing delays in these markets would have the greatest financial impact on the business.
* Conduct a detailed review of First Class and Second Class shipping processes. The exceptionally high delay rates suggest issues with carrier performance, routing efficiency, capacity planning, or unrealistic delivery commitments.
* Reassess planned delivery times and service-level agreements (SLAs). The consistently high delay rate indicates that current delivery estimates may not reflect actual operational capabilities.
* Investigate product categories with the highest delay rates to identify inventory shortages, warehouse processing issues, or supplier-related constraints. Targeted improvements in these categories could significantly reduce overall delays.
* Implement a regional logistics monitoring framework with monthly tracking of Delay Rate, Average Delay Days, and Late Sales Value. This would allow management to identify emerging issues earlier and measure the effectiveness of corrective actions.
* Focus resources on reducing severe delays rather than treating all delayed orders equally. Prioritizing high-value and high-impact delayed shipments can improve customer satisfaction and reduce financial losses more effectively.
##5.3 Summary
The analysis indicates that delivery delays are widespread across the supply chain, affecting over half of all orders. While delays occur across most regions, the greatest financial risk is concentrated in Western Europe and Central America. Shipping mode appears to be one of the strongest drivers of poor delivery performance, particularly for First Class and Second Class shipments. Targeted operational improvements in high-impact regions and transportation methods are likely to deliver the greatest business benefits.